# **Sentiment Analysis using NLP Pipeline & ML Models**

Dataset: IMDb Reviews (Kaggle)

---

## Objective

Build an end-to-end Sentiment Analysis system using:
- NLP preprocessing
- Feature Engineering (BoW & TF-IDF)
- Machine Learning Models

Compare model performance using evaluation metrics.

In [1]:
import pandas as pd
import numpy as np
import re
import string

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to C:\Users\SANJAY
[nltk_data]     M\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to C:\Users\SANJAY
[nltk_data]     M\AppData\Roaming\nltk_data...


True

In [3]:
df = pd.read_csv("IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


## 1. Data Understanding

In [5]:
print("Shape:", df.shape)
print("\nClass Distribution:\n", df['sentiment'].value_counts())

print("\nSample Reviews:\n")
print(df['review'].head())

Shape: (50000, 2)

Class Distribution:
 sentiment
positive    25000
negative    25000
Name: count, dtype: int64

Sample Reviews:

0    One of the other reviewers has mentioned that ...
1    A wonderful little production. <br /><br />The...
2    I thought this was a wonderful way to spend ti...
3    Basically there's a family where a little boy ...
4    Petter Mattei's "Love in the Time of Money" is...
Name: review, dtype: object


## 2. NLP Preprocessing

In [6]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    if not isinstance(text, str):
        return [], ""

    # Lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Remove numbers
    text = re.sub(r'\d+', '', text)

    # Tokenization
    tokens = text.split()

    # Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]

    # Lemmatization
    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    clean_text = " ".join(tokens)

    return tokens, clean_text

### APPLY PREPROCESSING

In [7]:
df['tokens'], df['clean_text'] = zip(*df['review'].apply(preprocess_text))

df[['review', 'clean_text']].head()

,review,clean_text
0,One of the other reviewers has mentioned that ...,one reviewer mentioned watching oz episode you...
1,A wonderful little production. <br /><br />The...,wonderful little production br br filming tech...
2,I thought this was a wonderful way to spend ti...,thought wonderful way spend time hot summer we...
3,Basically there's a family where a little boy ...,basically there family little boy jake think t...
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter matteis love time money visually stunni...


## 3. FEATURE ENGINEERING

### Convert text using Bag of Words (BoW)

In [8]:
bow = CountVectorizer(max_features=5000)
X_bow = bow.fit_transform(df['clean_text'])

### Convert text using TF-IDF.

In [9]:
tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(df['clean_text'])

## 4. MODEL BUILDING

### Target variable 

In [10]:
y = df['sentiment']

### Train test split

In [11]:
X_train_bow, X_test_bow, y_train, y_test = train_test_split(X_bow, y, test_size=0.2, random_state=42)
X_train_tfidf, X_test_tfidf, _, _ = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

### MODEL-1 (LOGISTIC REGRESSION)

In [12]:
lr = LogisticRegression()
lr.fit(X_train_tfidf, y_train)

y_pred_lr = lr.predict(X_test_tfidf)

### MODEL-2 (NAIVE BAYES)

In [13]:
nb = MultinomialNB()
nb.fit(X_train_bow, y_train)

y_pred_nb = nb.predict(X_test_bow)

### MODEL-3 (DECISION TREE)

In [14]:
dt = DecisionTreeClassifier()
dt.fit(X_train_bow, y_train)

y_pred_dt = dt.predict(X_test_bow)

## 5. MODEL EVALUATION

In [17]:
def evaluate_model(name, y_test, y_pred):
    print(f"\n{name}")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred, average='weighted'))
    print("Recall:", recall_score(y_test, y_pred, average='weighted'))
    print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))
    print("\nClassification Report:\n", classification_report(y_test, y_pred))

### EVALUATE ALL MODELS

In [19]:
evaluate_model("Logistic Regression (TF-IDF)", y_test, y_pred_lr)
evaluate_model("Naive Bayes (BoW)", y_test, y_pred_nb)
evaluate_model("Decision Tree (BoW)", y_test, y_pred_dt)


Logistic Regression (TF-IDF)
Accuracy: 0.8846
Precision: 0.8848144493161552
Recall: 0.8846
F1 Score: 0.8845710828086292

Classification Report:
               precision    recall  f1-score   support

    negative       0.89      0.87      0.88      4961
    positive       0.88      0.90      0.89      5039

    accuracy                           0.88     10000
   macro avg       0.88      0.88      0.88     10000
weighted avg       0.88      0.88      0.88     10000


Naive Bayes (BoW)
Accuracy: 0.8465
Precision: 0.8465388422559328
Recall: 0.8465
F1 Score: 0.8465022794998238

Classification Report:
               precision    recall  f1-score   support

    negative       0.84      0.85      0.85      4961
    positive       0.85      0.84      0.85      5039

    accuracy                           0.85     10000
   macro avg       0.85      0.85      0.85     10000
weighted avg       0.85      0.85      0.85     10000


Decision Tree (BoW)
Accuracy: 0.7115
Precision: 0.71153042
Recal

### COMPARISON TABLE

In [20]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Naive Bayes', 'Decision Tree'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_nb),
        accuracy_score(y_test, y_pred_dt)
    ],
    'Precision': [
        precision_score(y_test, y_pred_lr, average='weighted'),
        precision_score(y_test, y_pred_nb, average='weighted'),
        precision_score(y_test, y_pred_dt, average='weighted')
    ],
    'Recall': [
        recall_score(y_test, y_pred_lr, average='weighted'),
        recall_score(y_test, y_pred_nb, average='weighted'),
        recall_score(y_test, y_pred_dt, average='weighted')
    ],
    'F1 Score': [
        f1_score(y_test, y_pred_lr, average='weighted'),
        f1_score(y_test, y_pred_nb, average='weighted'),
        f1_score(y_test, y_pred_dt, average='weighted')
    ]
})

results

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.8846,0.884814,0.8846,0.884571
1,Naive Bayes,0.8465,0.846539,0.8465,0.846502
2,Decision Tree,0.7115,0.711530,0.7115,0.711504


## 6. Comparison & Insights


### 🔹 Best Preprocessing Steps
The best steps for text preprocessing, which resulted in the best outcome, are as follows:
- Lowercasing the text helped to normalize the text, removing duplicate words.
- Removing punctuation, numbers, and URLs helped to remove noise from the text.
- Removing stopwords improved the focus of the model on specific words.
- Using lemmatization resulted in better outcomes than stemming, as it helps retain the correct word meaning.
- Tokenization is necessary to transform the text correctly.

👉 Overall, the best outcome is achieved by using all the text preprocessing steps.

---

### 🔹 Best Vectorization Technique
The best technique for vectorizing the text is TF-IDF, which outperformed Bag of Words (BoW).
- TF-IDF is better than BoW because it considers the importance of words according to their occurrence in the entire document set.
- BoW only considers the occurrence of words without considering their importance.

👉 Therefore, the best technique for vectorizing the text is TF-IDF.

### 🔹 Best Model
- Logistic Regression has the highest Accuracy and F1 Score.
- It is good for dealing with large feature vectors and is sparse in nature, which is good for text data.
- It is not prone to overfitting, unlike Decision Trees.

👉 Best model combination:
- TF-IDF and Logistic Regression.

---

### 🔹 Trade-offs Between Models

1. Logistic Regression
   - Pros: High accuracy, more stable, and good for large feature vectors.
   - Cons: Slightly slower compared to Naive Bayes.

2. Naive Bayes
   - Pros: Very fast and good for text data.
   - Cons: Less accurate because of the assumption of independence.

3. Decision Tree
   - Pros: Easy to interpret and understand.
   - Cons: Prone to overfitting and not good for text data.

---

### 🔹 Final Conclusion
- Preprocessing is very important in improving the data.
- TF-IDF is the most effective feature extraction technique.
- Logistic Regression is the most efficient and effective model.
- Naive Bayes is faster but not as accurate as other models.

👉 Final Best Pipeline:
Preprocessing → TF-IDF → Logistic Regression

## Pipeline Flow

Raw Data → Preprocessing → Feature Engineering → Model Training → Evaluation → Comparison

### Testing

In [22]:
def predict_sentiment(text):
    _, clean = preprocess_text(text)
    vector = tfidf.transform([clean])
    return lr.predict(vector)[0]

print(predict_sentiment("This movie was amazing!"))

positive
